#### Imports

In [43]:
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

#### Loading Env

In [44]:
load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")

#### Loading PDF

In [45]:
pdf_path = "data/book.pdf"

loader = PyPDFLoader(pdf_path)
documents = loader.load()

print(f"Total pages loaded:{len(documents)}")

Total pages loaded:851


In [46]:
print('Sample Content:\n')
print(documents[0].page_content[:500])

print('\nMetadata:\n')
print(documents[0].metadata)

Sample Content:

Aurélien Géron
Hands-on  
Machine Learning  
 with Scikit-Learn,  
Keras & TensorFlow
Concepts, Tools, and Techniques  
to Build Intelligent Systems
TM
2nd Edition
Updated for  TensorFlow 2

Metadata:

{'producer': 'calibre (4.8.0) [https://calibre-ebook.com]', 'creator': 'calibre (4.8.0) [https://calibre-ebook.com]', 'creationdate': '2019-10-10T14:06:12+00:00', 'author': 'Aurélien Géron', 'ebx_publisher': "O'Reilly Media, Incorporated", 'moddate': '2020-01-16T12:03:44+00:00', 'title': 'Hands-on Machine Learning with Scikit-Learn, Keras, and TensorFlow', 'trapped': '/False', 'source': 'data/book.pdf', 'total_pages': 851, 'page': 0, 'page_label': 'Cover'}


#### Text Splitter

In [47]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size= 1800,
    chunk_overlap = 200,
    separators=["\n\n", "\n", " ", ""] 
)

split_docs = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(split_docs)}")
print(split_docs[2].page_content[:500])


Total chunks created: 1393
978-1-492-03264-9
[TI]
Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow
by Aurélien Géron
Copyright © 2019 Kiwisoft S.A.S. All rights reserved.
Printed in Canada.
Published by O’Reilly Media, Inc., 1005 Gravenstein Highway North, Sebastopol, CA 95472.
O’Reilly books may be purchased for educational, business, or sales promotional use. Online editions are
also available for most titles (http://oreilly.com). For more information, contact our corporate/institutional
sales departme


#### Embeddings

In [48]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name = "sentence-transformers/all-MiniLM-L6-v2"
)

sample_text = split_docs[0].page_content
embedding_vector = embedding_model.embed_query(sample_text)
print("Embedding Dimension:",len(embedding_vector))

Embedding Dimension: 384


#### Vector store

In [49]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents=split_docs,
    embedding= embedding_model
)

print("Vector store created")

vectorstore.save_local("faiss_index_langchain")

Vector store created


In [50]:
vectorstore = FAISS.load_local(
    "faiss_index_langchain",
    embedding_model,
    allow_dangerous_deserialization=True
)

#### Retriever

In [72]:
retriever = vectorstore.as_retriever(
    search_type = 'similarity',
    search_kwargs ={"k":5}
)

In [78]:
query = "What is overfitting?"

docs = retriever.invoke(query)
print(f"Retrieved {len(docs)} documents\n")

for i, doc in enumerate(docs):
    print(f"Document{i+1}")
    print(doc.page_content[:300])
    print("Page:", doc.metadata.get("page"))
    print()

Retrieved 5 documents

Document1
This is why it is called a trade-off.
Regularized Linear Models
As we saw in Chapters 1 and 2, a good way to reduce overfitting is to regularize the
model (i.e., to constrain it): the fewer degrees of freedom it has, the harder it will be
134 | Chapter 4: Training Models
Page: 163

Document2
Irrelevant Features
As the saying goes: garbage in, garbage out. Y our system will only be capable of learn‐
ing if the training data contains enough relevant features and not too many irrelevant
ones. A critical part of the success of a Machine Learning project is coming up with a
good set of featu
Page: 56

Document3
tem is at making predictions on the training data, plus a penalty for model com‐
plexity if the model is regularized. To make predictions, we feed the new
instance’s features into the model’s prediction function, using the parameter val‐
ues found by the learning algorithm.
14. Some of the main chal
Page: 749

Document4
Irrelevant Features            

#### Groq LLM Wrapper

In [79]:
from langchain_core.language_models import LLM
from groq import Groq

class GroqLLM(LLM):
    model: str = "llama-3.1-8b-instant"

    def _call(self, prompt, stop=None):
        client = Groq(api_key=groq_api_key)

        if isinstance(prompt,list):
            messages= prompt
        else:
            messages=[
                {'role':"user", 'content': prompt}
            ]

        response = client.chat.completions.create(
            model = self.model,
            messages = messages
        )

        return response.choices[0].message.content
    
    @property
    def _llm_type(self):
        return"groq"

In [80]:
from langchain_core.prompts import ChatPromptTemplate

prompt= ChatPromptTemplate.from_messages([
    ("system",
      "You are a strict RAG assistant. Answer ONLY from the provided context. "
     "If the answer is not found, say: 'I don't know based on the provided document.'" ),

    ("user",
      """Context:
        {context}

        Question:
        {question}

        Answer:
      """
    )
])

In [81]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

llm = GroqLLM()

def format_docs(docs):
    return "\n\n ".join(
        f"""
        Source: Page {doc.metadata.get('page')}

        Content:
        {doc.page_content}
        """
    for doc in docs
    )

rag_chain =(
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [83]:
query = "Explain relu in 5-6 points"

response = rag_chain.invoke(query)

print(response)

1. The ReLU (Rectified Linear Unit) function is a mathematical operation defined as max(0, z), which activates a neuron only when its input is greater than zero.
2. The ReLU function is continuous but not differentiable at z = 0, where the slope changes abruptly.
3. One of the major disadvantages of the ReLU function is that its derivative is zero for negative values of z, which can make Gradient Descent bounce around or get stuck.
4. However, ReLU has the advantage of being fast to compute, making it the default choice for many neural networks.
5. The ReLU function does not have a maximum output value, which helps reduce some issues during Gradient Descent.
6. Despite these limitations, ReLU has been shown to work very well in practice and is widely used in many neural network applications.
